In [1]:
# --- Standard library ---
import os
import re
import json
import uuid
import textwrap
import random
import time
from dataclasses import dataclass, field, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional, Tuple

# --- Third party ---
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Ensure no fallbacks ---
if not torch or not torch.cuda.is_available():
    raise RuntimeError("CUDA is required for this benchmark. Please install PyTorch with CUDA support.")

print("pandas:", pd.__version__)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA device:", torch.cuda.get_device_name(0))

/home/cc/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


pandas: 2.3.3
PyTorch version: 2.13.0+cu126
CUDA available: True
CUDA device: Tesla P100-PCIE-16GB


In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [3]:
# --- Enhanced Configuration ---
# Model configuration - Using a smaller model for stability
MODEL_NAME: str = "meta-llama/Llama-3.2-1B-Instruct"

# Generation parameters
MAX_NEW_TOKENS: int = 512
TEMPERATURE: float = 0.0  # Deterministic

# Agent configuration - Extended for comprehensive traces
MAX_AGENT_STEPS: int = 50  # More steps to reach ~100 events
MIN_EVENTS_PER_RUN: int = 90  # Ensure comprehensive traces
MAX_EVENTS_PER_RUN: int = 120

# Trace I/O configuration
TRACE_DIR: Path = Path("traces")
TRACE_DIR.mkdir(exist_ok=True)

# Summary configuration
SUMMARY_MAX_CHARS: int = 280

# LEP injection configuration
LEP_INJECTION_RATE: float = 0.3  # 30% chance of LEP injection
LEP_SEVERITY_DISTRIBUTION = {"low": 0.5, "medium": 0.3, "high": 0.2}

# ============================================================
# Multi-Agent System Configuration
# ============================================================

# Agent class definition with unique ID, name, role, and description
@dataclass
class AgentConfig:
    agent_id: str        # Unique identifier (e.g., "agent_001")
    name: str           # Display name (e.g., "researcher")
    role: str           # Job title/role (e.g., "Senior Research Analyst")
    description: str     # What this agent does
    capabilities: List[str]  # What tools this agent can use
    
# Define the agents in the system
AGENTS: List[AgentConfig] = [
    AgentConfig(
        agent_id="agent_001",
        name="researcher",
        role="Senior Research Analyst",
        description="Gathers, validates, and organizes information from various sources. Excels at finding specific data points and cross-referencing information.",
        capabilities=["read_document", "search_notes", "analyze_data"]
    ),
    AgentConfig(
        agent_id="agent_002",
        name="analyst",
        role="Financial Data Analyst",
        description="Specializes in calculations, financial analysis, and deriving insights from data. Works with numbers, metrics, and trend analysis.",
        capabilities=["calculator", "analyze_data", "coordinate_agents"]
    ),
]

# Create lookup dicts by agent_id and name
AGENT_BY_ID: Dict[str, AgentConfig] = {a.agent_id: a for a in AGENTS}
AGENT_BY_NAME: Dict[str, AgentConfig] = {a.name: a for a in AGENTS}

# Agent interaction protocols
AGENT_TEMPLATES: Dict[str, str] = {
    "researcher": """You are {agent_id} ({name}), a {role}.
{description}
You collaborate with other agents by sharing your findings and requesting specialized help when needed.
When another agent asks for information, provide it clearly and concisely.
When you need calculations or financial analysis, delegate to the analyst agent.""",
    
    "analyst": """You are {agent_id} ({name}), a {role}.
{description}
You collaborate with other agents by providing calculations, insights, and data interpretations.
When another agent provides data, perform appropriate analysis and share your findings.
When you need more information or research, delegate to the researcher agent.""",
}

# Task pool for comprehensive testing
TASK_POOL: List[str] = [
    "Read the financial report and calculate profit margins for Q1-Q4.",
    "Search meeting notes for any action items related to budget planning.",
    "Calculate the average revenue growth rate over the last 5 quarters.",
    "Read the project timeline document and identify key milestones.",
    "Search notes for dependencies between different projects.",
    "Calculate the ROI for marketing campaigns.",
    "Read risk assessment document and identify top 3 risks.",
    "Search for all decisions made in the last quarter.",
    "Calculate year-over-year growth for all product lines.",
    "Read competitive analysis and summarize key insights.",
]

# Tool ID prefix
TOOL_ID_PREFIX = "tool"

# Print configuration summary
print(f"MODEL_NAME         = {MODEL_NAME}")
print(f"TRACE_DIR          = {TRACE_DIR.resolve()}")
print(f"MAX_AGENT_STEPS    = {MAX_AGENT_STEPS}")
print(f"LEP_INJECTION_RATE = {LEP_INJECTION_RATE}")
print(f"Agents configured:")
for a in AGENTS:
    print(f"  {a.agent_id}: {a.name} ({a.role})")
    print(f"    Capabilities: {a.capabilities}")


MODEL_NAME         = meta-llama/Llama-3.2-1B-Instruct
TRACE_DIR          = /home/cc/traces
MAX_AGENT_STEPS    = 50
LEP_INJECTION_RATE = 0.3
Agents configured:
  agent_001: researcher (Senior Research Analyst)
    Capabilities: ['read_document', 'search_notes', 'analyze_data']
  agent_002: analyst (Financial Data Analyst)
    Capabilities: ['calculator', 'analyze_data', 'coordinate_agents']


In [4]:
# --- Robust LLM Loading with No Fallbacks ---
class LLMBackend:
    """Real LLM backend only - no fallbacks."""

    name: str = "real-llama"

    def __init__(self, model_name: str):
        self.model_name = model_name
        print(f"Loading model '{model_name}'...")
        
        # Load tokenizer with padding token
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        # Load model with explicit device map
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16,
            device_map="auto",
            trust_remote_code=True
        )
        self.model.eval()
        print("Model loaded successfully.")

    def _build_prompt(self, task, history, tools, agent_name="agent") -> str:
        # Add agent-specific role
        agent_template = AGENT_TEMPLATES.get(agent_name, "")
        tool_docs = "\n".join(f"- {n}: {t.description}" for n, t in tools.items())
        
        scratch = "\n".join(
            f"Step {i+1}: {h['agent']} called {h['tool']}({h['input']!r}) -> {h['output']}"
            for i, h in enumerate(history)
        ) or "(no tools called yet)"
        
        system = (
            f"{agent_template}\n\n"
            "You are a collaborative multi-agent system. Respond with ONLY a single JSON object, no prose.\n"
            "Schema: {\"reasoning\": str, \"action\": str, \"action_input\": str, \"final_response\": str}.\n"
            f"`action` must be one of: {list(tools.keys()) + ['final']}.\n"
            "Use 'final' (and fill final_response) when you can answer.\n"
            "Coordinate with other agents by sharing relevant information from previous steps.\n\n"
            f"Available tools:\n{tool_docs}"
        )
        user = f"Task: {task}\n\nScratchpad:\n{scratch}\n\nReturn the JSON action now."
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ]
        return self.tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    def _generate(self, prompt: str) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
        
        # Use generation parameters for better stability
        with torch.no_grad():
            out = self.model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=TEMPERATURE > 0,
                temperature=max(TEMPERATURE, 1e-4),
                pad_token_id=self.tokenizer.pad_token_id,
                eos_token_id=self.tokenizer.eos_token_id,
                repetition_penalty=1.1,  # Reduce repetition
                no_repeat_ngram_size=3,
            )
        new_tokens = out[0][inputs["input_ids"].shape[-1]:]
        return self.tokenizer.decode(new_tokens, skip_special_tokens=True)

    @staticmethod
    def _parse(raw: str, allowed: List[str]) -> Optional[Dict[str, Any]]:
        # Enhanced parsing to handle malformed JSON
        # Try to extract JSON from the raw output
        json_start = raw.find('{')
        json_end = raw.rfind('}') + 1
        
        if json_start == -1 or json_end == 0:
            return None
            
        json_str = raw[json_start:json_end]
        try:
            data = json.loads(json_str)
            # Validate required fields
            if "action" not in data or data["action"] not in allowed:
                return None
            return {
                "reasoning": str(data.get("reasoning", "")),
                "action": str(data["action"]),
                "action_input": str(data.get("action_input", "")),
                "final_response": str(data.get("final_response", ""))
            }
        except json.JSONDecodeError:
            return None

    def decide(self, task, history, tools, agent_name="agent") -> Dict[str, Any]:
        """Decide the next action using the real model."""
        allowed = list(tools.keys()) + ["final"]
        try:
            prompt = self._build_prompt(task, history, tools, agent_name)
            raw = self._generate(prompt)
            parsed = self._parse(raw, allowed)
            
            if parsed is not None:
                return parsed
            
            # If parsing fails, try a simpler prompt approach
            fallback_prompt = f"Task: {task}\nHistory: {history}\nAvailable tools: {allowed}\nRespond with JSON: {{\"action\": \"<tool_name>\"}}"
            raw = self._generate(fallback_prompt)
            parsed = self._parse(raw, allowed)
            
            if parsed is not None:
                return parsed
                
            # If still parsing fails, raise an error (no fallback)
            raise ValueError(f"Could not parse LLM output after retry. Raw output: {raw[:200]}")
            
        except Exception as err:
            # No fallback - raise the error
            raise RuntimeError(f"LLM generation failed: {err}") from err


def load_llm_backend() -> LLMBackend:
    """Load the real LLM backend - no fallbacks."""
    return LLMBackend(MODEL_NAME)


# Initialize the backend
print("Initializing real LLM backend...")
llm_backend = load_llm_backend()

Initializing real LLM backend...
Loading model 'meta-llama/Llama-3.2-1B-Instruct'...


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████████████████████████████████████████████████████████| 146/146 [00:00<00:00, 238.66it/s]


Model loaded successfully.


In [5]:
# --- Enhanced Tool Definitions with LEP Support ---
@dataclass
class Tool:
    """A callable tool with unique ID, name, description, and implementation."""
    tool_id: str        # Unique identifier (e.g., "tool_001")
    name: str          # Display name (e.g., "read_document")
    description: str  # What the tool does
    func: Callable[[str], Dict[str, Any]]  # Implementation
    
    def __call__(self, tool_input: str) -> Dict[str, Any]:
        return self.func(tool_input)

# Tool ID counter
_tool_counter = 0

def _make_tool_id() -> str:
    global _tool_counter
    _tool_counter += 1
    return f"tool_{_tool_counter:03d}"


# --- Enhanced knowledge base for more interactions ---
_DOCUMENTS: Dict[str, str] = {
    "financial_report": """
    Q1 Revenue: $1,200,000 | Q2 Revenue: $1,350,000 | Q3 Revenue: $1,500,000 | Q4 Revenue: $1,800,000
    Operating Costs: $800,000 | Marketing Spend: $200,000 | R&D Investment: $300,000
    Key Milestones: Product launch in Q1, Expansion in Q3, Partnership in Q4
    """,
    "market_analysis": """
    Market growth rate: 15% annually | Competitors: 5 major players
    Market share: 22% | Customer satisfaction: 4.6/5
    Growth opportunities: International expansion, Product diversification
    """,
    "project_timeline": """
    Phase 1 (Jan-Mar): Research and Planning
    Phase 2 (Apr-Jun): Development and Testing
    Phase 3 (Jul-Sep): Marketing and Launch
    Phase 4 (Oct-Dec): Review and Optimization
    """,
    "risk_assessment": """
    Financial risks: Market volatility, Currency fluctuations
    Operational risks: Supply chain delays, Resource constraints
    Strategic risks: Competition, Technology disruption
    Mitigation: Diversification, Partnerships, Innovation
    """,
    "default": "This is a reference document with general information."
}

_NOTES: List[str] = [
    "Team meeting: Q1 goals review and budget allocation decisions.",
    "Research finding: AI adoption increased by 40% in the last quarter.",
    "Action item: Prepare quarterly report for board meeting.",
    "Risk alert: Supply chain disruption affecting delivery timeline.",
    "Innovation opportunity: Explore AI-powered customer support solutions.",
    "Team sync: Cross-department collaboration on new product feature.",
    "Reminder: Submit patent application for new algorithm by next month.",
    "Market insight: Competitor launched similar product - differentiate value proposition."
]

_LEP_PATTERNS: List[Dict[str, Any]] = [
    {
        "pattern": r"(\d+)\s*\+\s*(\d+)",
        "replacement": r"\1+\2",  # Force addition
        "severity": "medium",
        "lep_type": "1.3 Step Repetition",
        "lep_category": "FC1"
    },
    {
        "pattern": r"calculate|compute|sum",
        "replacement": "ignore",
        "severity": "high",
        "lep_type": "2.3 Task Derailment", 
        "lep_category": "FC2"
    },
    {
        "pattern": r"document|report",
        "replacement": "skip",
        "severity": "low",
        "lep_type": "1.4 Loss of Conversation History",
        "lep_category": "FC1"
    }
]


# Enhanced tools with more complexity
def read_document(tool_input: str, lep_injected=False) -> Dict[str, Any]:
    """Enhanced document reader with multiple document support and LEP handling."""
    key = tool_input.strip().lower()
    
    # Extract document type
    if "financial" in key:
        doc_id = "financial_report"
    elif "market" in key:
        doc_id = "market_analysis" 
    elif "project" in key or "timeline" in key:
        doc_id = "project_timeline"
    elif "risk" in key:
        doc_id = "risk_assessment"
    else:
        doc_id = "default"
    
    content = _DOCUMENTS[doc_id]
    
    # Apply LEP if injected
    if lep_injected:
        for pattern in _LEP_PATTERNS:
            if "document" in pattern["pattern"].lower() or "report" in pattern["pattern"].lower():
                content = re.sub(pattern["pattern"], pattern["replacement"], content)
                break
    
    # Generate additional interactions for longer traces
    if "calculate" in key or "compute" in key:
        # Try to extract numbers for potential calculations
        numbers = re.findall(r'\d+', content)
        if numbers:
            content += f" | Available numbers: {', '.join(numbers)}"
    
    return {
        "output": content,
        "metadata": {
            "doc_id": doc_id,
            "char_len": len(content),
            "lep_injected": lep_injected
        }
    }


def search_notes(tool_input: str, lep_injected=False) -> Dict[str, Any]:
    """Enhanced note search with multiple query support and LEP handling."""
    query = tool_input.strip().lower()
    
    # Tokenize query
    tokens = [t for t in re.split(r"\W+", query) if len(t) > 2]
    
    # Apply LEP if injected
    if lep_injected:
        # Randomly modify some notes
        modified_notes = []
        for note in _NOTES:
            if random.random() < 0.3:  # 30% chance to modify
                for pattern in _LEP_PATTERNS:
                    note = re.sub(pattern["pattern"], pattern["replacement"], note)
            modified_notes.append(note)
    else:
        modified_notes = _NOTES
    
    # Find matches
    hits = [n for n in modified_notes if any(t in n.lower() for t in tokens)]
    
    if not hits:
        hits = [n for n in modified_notes if any(t in n.lower() for t in query.split())]
    
    # Limit results but ensure we get some
    hits = hits[:5] or modified_notes[:2]
    
    return {
        "output": " | ".join(hits),
        "metadata": {
            "query": query,
            "num_hits": len(hits),
            "lep_injected": lep_injected
        }
    }


def calculator(tool_input: str, lep_injected=False) -> Dict[str, Any]:
    """Enhanced calculator with expression validation and LEP injection."""
    runs = re.findall(r"[0-9.\+\-\*/()\s]+", tool_input)
    candidates = [r.strip() for r in runs if re.search(r"[-+*/]", r) and re.search(r"\d", r)]
    expr = max(candidates, key=len) if candidates else ""
    
    # Store original for comparison
    original_expr = expr
    
    # Apply LEP if injected
    if lep_injected:
        for pattern in _LEP_PATTERNS:
            if re.search(r"\d+\s*\+\s*\d+", expr):
                if pattern["severity"] == "medium":
                    # Force simple addition
                    numbers = re.findall(r'\d+', expr)
                    if len(numbers) >= 2:
                        expr = f"{numbers[0]}+{numbers[1]}"
                elif pattern["severity"] == "high":
                    # Skip calculation
                    expr = "0"
    
    try:
        # Restricted eval: only arithmetic operators
        result = eval(expr, {"__builtins__": {}}, {}) if expr else "no expression found"
    except Exception as err:
        result = f"error: {err}"
    
    return {
        "output": str(result),
        "metadata": {
            "expression": expr,
            "original_expression": original_expr,
            "lep_injected": lep_injected,
            "computation_valid": str(result) not in ["error: no expression found", "0"]
        }
    }


def analyze_data(tool_input: str, lep_injected=False) -> Dict[str, Any]:
    """New tool for data analysis - generates more events."""
    query = tool_input.strip().lower()
    
    # Extract context from previous documents
    analysis = ""
    if "revenue" in query or "financial" in query:
        analysis = "Revenue analysis shows steady growth: Q1: $1.2M, Q2: $1.35M, Q3: $1.5M, Q4: $1.8M"
        if lep_injected:
            analysis = analysis.replace("1.8M", "1.0M")  # Inject error
    
    elif "market" in query:
        analysis = "Market position: 22% market share, 4.6/5 customer satisfaction"
        if lep_injected:
            analysis = analysis.replace("22%", "5%")  # Severe understatement
    
    elif "growth" in query:
        growth_rate = "15%"
        if lep_injected:
            growth_rate = "1%"  # Inject pessimistic outlook
        analysis = f"Growth rate: {growth_rate} annually with strong competitive positioning"
    
    # Add insights based on available data
    if "trend" in query:
        trends = ["Increasing market share", "Strong revenue growth", "Positive customer feedback"]
        analysis += f". Trends: {', '.join(trends)}"
    
    return {
        "output": analysis,
        "metadata": {
            "query_type": query,
            "lep_injected": lep_injected,
            "analysis_depth": "comprehensive"
        }
    }


def coordinate_agents(tool_input: str, lep_injected=False) -> Dict[str, Any]:
    """New tool for inter-agent coordination."""
    # Simulate agent-to-agent communication
    coordination_notes = [
        "Researcher provided Q4 market analysis data",
        "Analyst requested financial projections for planning", 
        "Shared timeline dependencies between teams",
        "Cross-functional alignment on deliverables",
        "Risk assessment updated with new market information"
    ]
    
    if lep_injected:
        # Inject miscommunication
        coordination_notes.append("Missing critical data point in analysis")
        coordination_notes.append("Inconsistent information shared between agents")
    
    return {
        "output": "Coordination events: " + " | ".join(coordination_notes),
        "metadata": {
            "agents_involved": len(AGENTS),
            "lep_injected": lep_injected,
            "communication_quality": "reduced" if lep_injected else "high"
        }
    }


# Registry for all tools
ENHANCED_TOOLS: Dict[str, Tool] = {
    "read_document": Tool(
        _make_tool_id(), "read_document", "Read a named document by keyword (financial, market, project, risk).", 
        lambda x, lep_injected=False: read_document(x, lep_injected=lep_injected)
    ),
    "search_notes": Tool(
        _make_tool_id(), "search_notes", "Search internal notes for query terms.", 
        lambda x, lep_injected=False: search_notes(x, lep_injected=lep_injected)
    ),
    "calculator": Tool(
        _make_tool_id(), "calculator", "Evaluate arithmetic expressions with validation.", 
        lambda x, lep_injected=False: calculator(x, lep_injected=lep_injected)
    ),
    "analyze_data": Tool(
        _make_tool_id(), "analyze_data", "Analyze data patterns and trends from documents.", 
        lambda x, lep_injected=False: analyze_data(x, lep_injected=lep_injected)
    ),
    "coordinate_agents": Tool(
        _make_tool_id(), "coordinate_agents", "Coordinate communication between agents.", 
        lambda x, lep_injected=False: coordinate_agents(x, lep_injected=lep_injected)
    ),
}

# Tool lookup by ID
TOOL_BY_ID: Dict[str, Tool] = {t.tool_id: t for t in ENHANCED_TOOLS.values()}

print("Enhanced tools loaded with LEP support:")
for name, tool in ENHANCED_TOOLS.items():
    print(f"  - {name}: {tool.description}")

Enhanced tools loaded with LEP support:
  - read_document: Read a named document by keyword (financial, market, project, risk).
  - search_notes: Search internal notes for query terms.
  - calculator: Evaluate arithmetic expressions with validation.
  - analyze_data: Analyze data patterns and trends from documents.
  - coordinate_agents: Coordinate communication between agents.


## 5. TraceCollector Implementation

The heart of the project. `TraceEvent` is a dataclass mirroring the exact JSONL schema. The
labeling fields fall into four groups, all of which default to empty/`None` and are **never**
set by the collector itself — only later labeling may set them:

- **LEP labeling**: `lep_injected`, `lep_type`, `lep_category`, `lep_location`, `lep_severity`
- **forensic**: `risk_tags`
- **causality / dependency**: `caused_by_event`, `depends_on`, `propagates_to`
- **failure outcome**: `downstream_failure`, `failure_type`, `failure_event`

The factual `expected_behavior` / `observed_behavior` fields may be filled at log time.

`TraceCollector` manages a single append-only `.jsonl` file:

- `start_trace()` — begin a new trace (fresh `trace_id`, reset event counter, open a file).
- `log_event(...)` — build one event and **immediately append** it to disk as one line.
- `save_jsonl()` — (re)write the full in-memory trace to disk in one shot.
- `load_jsonl()` — read a `.jsonl` file back into a list of event dicts.
- `pretty_print_trace()` — human-readable console rendering.

In [6]:
# The set of valid event types (kept as a constant for documentation + light validation).
EVENT_TYPES = (
    "user_input",
    "system_init",
    "agent_handoff",
    "reasoning",
    "tool_call",
    "tool_result",
    "llm_output",
    "final_response",
)


def _utc_now_iso() -> str:
    """ISO-8601 UTC timestamp (sortable, timezone-aware)."""
    return datetime.now(timezone.utc).isoformat()


def _summarize(value: Any, limit: int = SUMMARY_MAX_CHARS) -> str:
    """Coerce a value to a trimmed single-line string for *_summary fields."""
    text = value if isinstance(value, str) else json.dumps(value, default=str)
    text = " ".join(text.split())  # collapse whitespace/newlines
    return text if len(text) <= limit else text[: limit - 1] + "\u2026"


@dataclass
class TraceEvent:
    """One execution event == one JSON line. Mirrors the required JSONL schema exactly.

    Beyond the raw facts (source/target/summaries), the schema carries fields designed to
    support *future* graph analysis of Localized Execution Perturbations (LEPs) and how they
    propagate into downstream failures:

    - behavior (`expected_behavior`, `observed_behavior`): factual descriptions; may be filled
      by the agent at log time.
    - LEP labeling (`lep_injected`, `lep_type`, `lep_category`, `lep_location`, `lep_severity`):
      set ONLY during later labeling/injection experiments — never by the collector.
    - causality / dependency (`caused_by_event`, `depends_on`, `propagates_to`): the edges that
      let us reconstruct a propagation graph; also labeling-only.
    - failure outcome (`downstream_failure`, `failure_type`, `failure_event`): whether/how this
      event leads to a failure, and which event id the failure manifests at; labeling-only.
    """

    trace_id: str
    event_id: int
    timestamp: str
    event_type: str
    source: str
    target: str
    input_summary: str
    output_summary: str

    # --- Agent context ---
    agent_id: Optional[str] = None       # e.g. "agent_001"
    agent_name: Optional[str] = None      # e.g. "researcher"
    agent_role: Optional[str] = None      # e.g. "Senior Research Analyst"
    
    # --- Tool context ---
    tool_id: Optional[str] = None        # e.g. "tool_001"
    tool_name: Optional[str] = None      # e.g. "calculator"
    
    # --- Behavior description (factual; may be filled by the agent at log time) ---
    expected_behavior: str = ""
    observed_behavior: str = ""

    # --- LEP labeling: for FUTURE labeling/injection only; collector never sets these. ---
    lep_injected: bool = False            # was a perturbation deliberately injected here?
    lep_type: Optional[str] = None        # e.g. "1.3 Step Repetition"
    lep_category: Optional[str] = None    # e.g. "FC1"
    lep_location: Optional[str] = None    # where the perturbation manifests
    lep_severity: Optional[str] = None    # e.g. "low" | "medium" | "high"

    # --- Forensic ---
    risk_tags: List[str] = field(default_factory=list)

    # --- Causality / dependency: edges of the LEP propagation graph (labeling-only). ---
    caused_by_event: Optional[int] = None                    # event_id that caused this one
    depends_on: List[int] = field(default_factory=list)      # event_ids this one depends on
    propagates_to: List[int] = field(default_factory=list)   # event_ids this propagates into

    # --- Handoff tracking ---
    agent_id_from: Optional[str] = None     # agent handing off
    agent_name_from: Optional[str] = None
    agent_id_to: Optional[str] = None       # agent receiving control
    agent_name_to: Optional[str] = None
    
    # --- Failure outcome (labeling-only). ---
    downstream_failure: bool = False          # did this lead to a downstream failure?
    failure_type: Optional[str] = None        # e.g. "non_termination" | "wrong_answer"
    failure_event: Optional[int] = None       # event_id where the failure manifests

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)


class TraceCollector:
    """Records agent execution as an append-only JSONL trace.

    The collector only records *facts*. It makes no judgement about whether a trace is
    normal or abnormal.
    """

    def __init__(self, trace_dir: Path = TRACE_DIR):
        self.trace_dir = Path(trace_dir)
        self.trace_dir.mkdir(exist_ok=True)
        self.trace_id: Optional[str] = None
        self.path: Optional[Path] = None
        self._next_event_id: int = 1
        self.events: List[TraceEvent] = []

    # --- lifecycle ---------------------------------------------------------
    def start_trace(self, trace_id: Optional[str] = None) -> str:
        """Begin a new trace; returns the trace_id and opens a fresh JSONL file."""
        self.trace_id = trace_id or uuid.uuid4().hex[:12]
        self._next_event_id = 1
        self.events = []
        self.path = self.trace_dir / f"trace_{self.trace_id}.jsonl"
        # Truncate/create the file so each trace starts clean.
        self.path.write_text("", encoding="utf-8")
        return self.trace_id

    # --- writing -----------------------------------------------------------
    def log_event(
        self,
        event_type: str,
        source: str,
        target: str,
        input_summary: Any = "",
        output_summary: Any = "",
        *,
        # Agent context
        agent_id: Optional[str] = None,
        agent_name: Optional[str] = None,
        agent_role: Optional[str] = None,
        # Tool context
        tool_id: Optional[str] = None,
        tool_name: Optional[str] = None,
        # Handoff tracking
        agent_id_from: Optional[str] = None,
        agent_name_from: Optional[str] = None,
        agent_id_to: Optional[str] = None,
        agent_name_to: Optional[str] = None,
        # Behavior
        expected_behavior: str = "",
        observed_behavior: str = "",
        # LEP
        lep_injected: bool = False,
        lep_type: Optional[str] = None,
        lep_category: Optional[str] = None,
        lep_location: Optional[str] = None,
        lep_severity: Optional[str] = None,
        # Forensic
        risk_tags: Optional[List[str]] = None,
        # Causality
        caused_by_event: Optional[int] = None,
        depends_on: Optional[List[int]] = None,
        propagates_to: Optional[List[int]] = None,
        # Failure
        downstream_failure: bool = False,
        failure_type: Optional[str] = None,
        failure_event: Optional[int] = None,
    ) -> TraceEvent:
        """Create one event and immediately append it to disk as a single JSONL line.

        Only `expected_behavior` / `observed_behavior` are expected to be set during normal
        runs. All LEP / causality / failure fields default to empty and are intended to be
        filled in later by labeling tools (see `annotate_lep` / `link_propagation`).
        """
        if self.trace_id is None or self.path is None:
            raise RuntimeError("Call start_trace() before log_event().")
        if event_type not in EVENT_TYPES:
            raise ValueError(f"Unknown event_type {event_type!r}; expected one of {EVENT_TYPES}")

        event = TraceEvent(
            trace_id=self.trace_id,
            event_id=self._next_event_id,
            timestamp=_utc_now_iso(),
            event_type=event_type,
            source=source,
            target=target,
            input_summary=_summarize(input_summary),
            output_summary=_summarize(output_summary),
            # Agent context
            agent_id=agent_id,
            agent_name=agent_name,
            agent_role=agent_role,
            # Tool context
            tool_id=tool_id,
            tool_name=tool_name,
            # Behavior
            expected_behavior=_summarize(expected_behavior),
            observed_behavior=_summarize(observed_behavior),
            # Handoff
            agent_id_from=agent_id_from,
            agent_name_from=agent_name_from,
            agent_id_to=agent_id_to,
            agent_name_to=agent_name_to,
            # LEP
            lep_injected=lep_injected,
            lep_type=lep_type,
            lep_category=lep_category,
            lep_location=lep_location,
            lep_severity=lep_severity,
            # Forensic
            risk_tags=list(risk_tags) if risk_tags else [],
            # Causality
            caused_by_event=caused_by_event,
            depends_on=list(depends_on) if depends_on else [],
            propagates_to=list(propagates_to) if propagates_to else [],
            # Failure
            downstream_failure=downstream_failure,
            failure_type=failure_type,
            failure_event=failure_event,
        )
        self._next_event_id += 1
        self.events.append(event)

        # Append-only write: one JSON object per line, flushed immediately.
        with self.path.open("a", encoding="utf-8") as fh:
            fh.write(json.dumps(event.to_dict(), default=str) + "\n")
        return event

    # --- persistence -------------------------------------------------------
    def save_jsonl(self, path: Optional[Path] = None) -> Path:
        """Rewrite the full in-memory trace to disk in one shot (idempotent snapshot)."""
        target = Path(path) if path else self.path
        if target is None:
            raise RuntimeError("No path to save to; start a trace first.")
        with target.open("w", encoding="utf-8") as fh:
            for event in self.events:
                fh.write(json.dumps(event.to_dict(), default=str) + "\n")
        return target

    @staticmethod
    def load_jsonl(path: Path) -> List[Dict[str, Any]]:
        """Load a JSONL trace file into a list of event dicts."""
        rows: List[Dict[str, Any]] = []
        with Path(path).open("r", encoding="utf-8") as fh:
            for line in fh:
                line = line.strip()
                if line:
                    rows.append(json.loads(line))
        return rows

    # --- display -----------------------------------------------------------
    def pretty_print_trace(self) -> None:
        """Render the current trace to the console in a readable, ordered format."""
        print(f"=== Trace {self.trace_id} ({len(self.events)} events) ===")
        for ev in self.events:
            print(
                f"[{ev.event_id:>2}] {ev.event_type:<14} "
                f"{ev.source} -> {ev.target}"
            )
            if ev.input_summary:
                print(f"      in : {ev.input_summary}")
            if ev.output_summary:
                print(f"      out: {ev.output_summary}")


print("TraceCollector ready. Event types:", EVENT_TYPES)

TraceCollector ready. Event types: ('user_input', 'system_init', 'agent_handoff', 'reasoning', 'tool_call', 'tool_result', 'llm_output', 'final_response')


In [7]:
# --- Localized Execution Perturbation (LEP) taxonomy -------------------------
# Three failure categories, each mapping LEP code -> human-readable definition.
LEP_TAXONOMY: Dict[str, Dict[str, Any]] = {
    "FC1": {
        "name": "System Design Issues",
        "description": "Failures originating from system design decisions, prompt "
                       "specifications, or state management.",
        "leps": {
            "1.1": "Disobey Task Specification",
            "1.2": "Disobey Role Specification",
            "1.3": "Step Repetition",
            "1.4": "Loss of Conversation History",
            "1.5": "Unaware of Termination Conditions",
        },
    },
    "FC2": {
        "name": "Inter-Agent Misalignment",
        "description": "Failures arising from breakdowns in information flow and coordination "
                       "between agents.",
        "leps": {
            "2.1": "Conversation Reset",
            "2.2": "Fail to Ask for Clarification",
            "2.3": "Task Derailment",
            "2.4": "Information Withholding",
            "2.5": "Ignored Other Agent's Input",
            "2.6": "Reasoning-Action Mismatch",
        },
    },
    "FC3": {
        "name": "Task Verification",
        "description": "Failures involving inadequate verification or quality control.",
        "leps": {
            "3.1": "Premature Termination",
            "3.2": "No or Incomplete Verification",
            "3.3": "Incorrect Verification",
        },
    },
}

# Flat lookups: "1.3" -> "FC1" and "1.3" -> "Step Repetition".
LEP_CODE_TO_CATEGORY: Dict[str, str] = {
    code: cat for cat, spec in LEP_TAXONOMY.items() for code in spec["leps"]
}
LEP_CODE_TO_NAME: Dict[str, str] = {
    code: name for spec in LEP_TAXONOMY.values() for code, name in spec["leps"].items()
}


def _find_event(events: List[Dict[str, Any]], event_id: int) -> Dict[str, Any]:
    for ev in events:
        if ev["event_id"] == event_id:
            return ev
    raise KeyError(f"event_id {event_id} not found in trace")


def annotate_lep(
    events: List[Dict[str, Any]],
    event_id: int,
    lep_code: str,
    *,
    lep_location: Optional[str] = None,
    lep_severity: Optional[str] = None,
    lep_injected: bool = False,
    downstream_failure: bool = False,
    failure_type: Optional[str] = None,
    failure_event: Optional[int] = None,
) -> Dict[str, Any]:
    """Label one event with an LEP. `lep_category` is auto-derived from the taxonomy.

    `lep_code` is a taxonomy code such as "1.3". The stored `lep_type` is "1.3 Step Repetition".
    `lep_injected` marks deliberately injected perturbations; `lep_severity` is a free-form
    severity ("low"/"medium"/"high"). When the event culminates in a failure, set
    `downstream_failure` plus `failure_type` / `failure_event`. Returns the mutated event dict.
    """
    if lep_code not in LEP_CODE_TO_CATEGORY:
        raise ValueError(f"Unknown LEP code {lep_code!r}; valid codes: {list(LEP_CODE_TO_CATEGORY)}")
    ev = _find_event(events, event_id)
    ev["lep_injected"] = lep_injected
    ev["lep_type"] = f"{lep_code} {LEP_CODE_TO_NAME[lep_code]}"
    ev["lep_category"] = LEP_CODE_TO_CATEGORY[lep_code]
    ev["lep_location"] = lep_location if lep_location is not None else ev.get("event_type")
    ev["lep_severity"] = lep_severity
    ev["downstream_failure"] = downstream_failure
    ev["failure_type"] = failure_type
    ev["failure_event"] = failure_event
    return ev


def link_propagation(
    events: List[Dict[str, Any]],
    cause_id: int,
    effect_ids: List[int],
) -> None:
    """Record causal propagation: cause -> effects.

    Adds each effect id to the cause's `propagates_to`, sets each effect's `caused_by_event`
    to the cause id, and records the reverse `depends_on` edge on each effect — together
    forming the directed edges of the propagation graph.
    """
    cause = _find_event(events, cause_id)
    cause.setdefault("propagates_to", [])
    for eid in effect_ids:
        effect = _find_event(events, eid)
        if eid not in cause["propagates_to"]:
            cause["propagates_to"].append(eid)
        effect["caused_by_event"] = cause_id
        effect.setdefault("depends_on", [])
        if cause_id not in effect["depends_on"]:
            effect["depends_on"].append(cause_id)


print("LEP taxonomy loaded:",
      {cat: len(spec["leps"]) for cat, spec in LEP_TAXONOMY.items()})

LEP taxonomy loaded: {'FC1': 5, 'FC2': 6, 'FC3': 3}


In [8]:
class Agent:
    """A lightweight reason -> act -> observe loop that records a full trace."""

    def __init__(
        self,
        llm: LLMBackend,
        tools: Dict[str, Tool],
        collector: TraceCollector,
        max_steps: int = MAX_AGENT_STEPS,
    ):
        self.llm = llm
        self.tools = tools
        self.collector = collector
        self.max_steps = max_steps

    def run(self, task: str) -> Dict[str, Any]:
        """Execute one task end-to-end, emitting trace events at every step."""
        trace_id = self.collector.start_trace()

        # 1) The user hands a task to the agent.
        self.collector.log_event(
            event_type="user_input",
            source="user",
            target="agent",
            input_summary=task,
            output_summary="",
            expected_behavior="User submits a well-formed task for the agent to solve.",
            observed_behavior=f"User submitted task: {task}",
        )

        history: List[Dict[str, Any]] = []  # observations fed back into the LLM
        final_response = ""

        for step in range(1, self.max_steps + 1):
            # 2) LLM reasons about the next step.
            decision = self.llm.decide(task, history, self.tools)
            self.collector.log_event(
                event_type="reasoning",
                source="llm",
                target="internal",
                input_summary=task,
                output_summary=decision.reasoning,
                expected_behavior="Agent reasons toward the task goal and picks a valid next action.",
                observed_behavior=f"Agent reasoned and chose action '{decision.action}'.",
            )

            # 3) Terminal branch: the LLM decided to answer.
            if decision.action == "final":
                final_response = decision.final_response or "(no answer produced)"
                # llm_output captures the raw model decision to finalize...
                self.collector.log_event(
                    event_type="llm_output",
                    source="llm",
                    target="agent",
                    input_summary=task,
                    output_summary=final_response,
                    expected_behavior="Agent produces a final answer grounded in the gathered observations.",
                    observed_behavior="Agent emitted a final answer.",
                )
                # ...and final_response is the answer handed back to the user.
                self.collector.log_event(
                    event_type="final_response",
                    source="agent",
                    target="user",
                    input_summary=task,
                    output_summary=final_response,
                    expected_behavior="Agent returns a correct, complete answer to the user.",
                    observed_behavior="Agent returned the final answer to the user.",
                )
                break

            # 4) Tool branch: validate, log the call, execute, log the result.
            tool = self.tools.get(decision.action)
            if tool is None:
                # Unknown action -> record it as reasoning and stop to stay safe.
                self.collector.log_event(
                    event_type="reasoning",
                    source="llm",
                    target="internal",
                    input_summary=decision.action,
                    output_summary=f"Unknown action '{decision.action}'; aborting loop.",
                    expected_behavior="Agent selects a registered tool or 'final'.",
                    observed_behavior=f"Agent selected unknown action '{decision.action}'.",
                )
                final_response = f"Could not complete task: unknown action {decision.action!r}."
                self.collector.log_event(
                    event_type="final_response", source="agent", target="user",
                    input_summary=task, output_summary=final_response,
                    expected_behavior="Agent returns a correct, complete answer to the user.",
                    observed_behavior="Agent aborted due to an unknown action.",
                )
                break

            self.collector.log_event(
                event_type="tool_call",
                source="agent",
                target=tool.name,
                input_summary=decision.action_input,
                output_summary="",
                expected_behavior=f"Agent invokes '{tool.name}' with an input appropriate for the task.",
                observed_behavior=f"Agent called '{tool.name}' with input: {decision.action_input}",
            )

            result = tool(decision.action_input)
            self.collector.log_event(
                event_type="tool_result",
                source=tool.name,
                target="agent",
                input_summary=decision.action_input,
                output_summary=result.get("output", ""),
                expected_behavior=f"'{tool.name}' returns a correct, usable result.",
                observed_behavior=f"'{tool.name}' returned: {result.get('output', '')}",
            )

            history.append({
                "tool": tool.name,
                "input": decision.action_input,
                "output": result.get("output", ""),
            })
        else:
            # Loop exhausted without finalizing -> emit a fallback final_response.
            final_response = "Reached max steps without a final answer."
            self.collector.log_event(
                event_type="final_response",
                source="agent",
                target="user",
                input_summary=task,
                output_summary=final_response,
                expected_behavior="Agent recognizes task completion and returns the answer within the step budget.",
                observed_behavior="Agent exhausted its step budget without producing a final answer.",
            )

        return {
            "trace_id": trace_id,
            "final_response": final_response,
            "num_events": len(self.collector.events),
            "path": str(self.collector.path),
        }


print("Agent ready.")

Agent ready.


In [9]:
# --- Enhanced Multi-Agent Implementation ---

class MultiAgentSystem:
    """Multi-agent system with true agent collaboration and comprehensive trace generation."""

    def __init__(self, llm_backend, tools, trace_dir=TRACE_DIR):
        self.llm_backend = llm_backend
        self.tools = tools
        self.trace_dir = Path(trace_dir)
        self.trace_dir.mkdir(exist_ok=True)

    def _should_inject_lep(self) -> bool:
        """Determine if LEP should be injected based on configuration."""
        return random.random() < LEP_INJECTION_RATE

    def _get_lep_severity(self) -> str:
        """Get LEP severity based on configured distribution."""
        rand = random.random()
        cumulative = 0.0
        for severity, prob in LEP_SEVERITY_DISTRIBUTION.items():
            cumulative += prob
            if rand <= cumulative:
                return severity
        return "low"

    def _log_agent_event(self, collector, event_type: str, source_agent: AgentConfig,
                         target: str, input_summary: str, output_summary: str,
                         expected_behavior: str, observed_behavior: str,
                         lep_injected: bool = False, **kwargs):
        """Log an event with proper agent_id and agent_name."""
        collector.log_event(
            event_type=event_type,
            source=source_agent.agent_id,
            target=target,
            input_summary=input_summary,
            output_summary=output_summary,
            expected_behavior=expected_behavior,
            observed_behavior=observed_behavior,
            agent_name=source_agent.name,
            agent_role=source_agent.role,
            lep_injected=lep_injected,
            **kwargs
        )

    def _log_tool_event(self, collector, event_type: str, source_agent: AgentConfig,
                        tool: Tool, input_summary: str, output_summary: str,
                        expected_behavior: str, observed_behavior: str,
                        lep_injected: bool = False, **kwargs):
        """Log a tool event with proper tool_id."""
        collector.log_event(
            event_type=event_type,
            source=source_agent.agent_id,
            target=tool.tool_id,
            input_summary=input_summary,
            output_summary=output_summary,
            expected_behavior=expected_behavior,
            observed_behavior=observed_behavior,
            agent_id=source_agent.agent_id,
            agent_name=source_agent.name,
            tool_id=tool.tool_id,
            tool_name=tool.name,
            lep_injected=lep_injected,
            **kwargs
        )

    def _generate_benign_trace(self, task: str, run_id: str) -> Dict[str, Any]:
        """Generate a trace without LEP injection."""
        collector = TraceCollector()
        trace_id = f"{run_id}_benign"
        collector.start_trace(trace_id)
        return self._execute_collaborative_task(collector, task, run_id, "benign")

    def _generate_malignant_trace(self, task: str, run_id: str) -> Dict[str, Any]:
        """Generate a trace with LEP injection."""
        collector = TraceCollector()
        trace_id = f"{run_id}_malignant"
        collector.start_trace(trace_id)
        return self._execute_collaborative_task(collector, task, run_id, "malignant")

    def _execute_collaborative_task(self, collector, task: str, run_id: str, trace_type: str) -> Dict[str, Any]:
        """Execute task with true multi-agent collaboration."""

        collector.log_event(
            event_type="user_input",
            source="user",
            target="multi_agent_system",
            input_summary=f"Run ID: {run_id} | Task: {task}",
            output_summary="",
            expected_behavior="Multi-agent system receives collaborative task.",
            observed_behavior=f"System received task: {task}",
        )

        primary_agent = AGENT_BY_NAME["researcher"]
        secondary_agent = AGENT_BY_NAME["analyst"]

        collector.log_event(
            event_type="system_init",
            source="system",
            target="multi_agent_system",
            input_summary="Initialize multi-agent collaboration",
            output_summary=f"Primary: {primary_agent.agent_id} ({primary_agent.name}) | Secondary: {secondary_agent.agent_id} ({secondary_agent.name})",
            expected_behavior="Both agents initialized and ready for collaboration.",
            observed_behavior=f"Agents ready: {primary_agent.name}, {secondary_agent.name}",
        )

        history = []
        final_response = ""
        lep_injected_this_run = False
        should_inject_leps = (trace_type == "malignant")
        handoff_count = 0

        for step in range(1, MAX_AGENT_STEPS + 1):
            if step >= MIN_EVENTS_PER_RUN and len(collector.events) >= MIN_EVENTS_PER_RUN:
                if len(collector.events) >= MAX_EVENTS_PER_RUN or step > MIN_EVENTS_PER_RUN + 10:
                    break

            needs_handoff = (
                ("calculate" in task.lower() or "roi" in task.lower() or "profit" in task.lower()) and step > 3
            ) or (step > 15 and step % 5 == 0)

            if needs_handoff and handoff_count < 3:
                current_agent = secondary_agent if primary_agent.name == "researcher" else primary_agent
                handoff_count += 1

                collector.log_event(
                    event_type="agent_handoff",
                    source=primary_agent.agent_id,
                    target=current_agent.agent_id,
                    input_summary=f"Handoff from {primary_agent.name} to {current_agent.name}",
                    output_summary=f"{primary_agent.name} handing off to {current_agent.name}",
                    expected_behavior=f"Control transfers from {primary_agent.role} to {current_agent.role}.",
                    observed_behavior=f"Handoff: {primary_agent.agent_id} -> {current_agent.agent_id}",
                    agent_id_from=primary_agent.agent_id,
                    agent_name_from=primary_agent.name,
                    agent_id_to=current_agent.agent_id,
                    agent_name_to=current_agent.name,
                )
            else:
                current_agent = primary_agent

            lep_injected = should_inject_leps and self._should_inject_lep() and step > 3
            lep_injected_this_run = lep_injected_this_run or lep_injected

            reasoning_content = self._generate_reasoning_content(task, step, history, current_agent)

            collector.log_event(
                event_type="reasoning",
                source=current_agent.agent_id,
                target="internal",
                input_summary=f"Task: {task} | Step: {step}",
                output_summary=reasoning_content,
                expected_behavior=f"{current_agent.role} reasons about next action.",
                observed_behavior=f"{current_agent.agent_id} ({current_agent.name}) reasoning.",
                agent_id=current_agent.agent_id,
                agent_name=current_agent.name,
                agent_role=current_agent.role,
                lep_injected=lep_injected,
                lep_type="1.3 Step Repetition" if lep_injected else None,
                lep_category="FC1" if lep_injected else None,
                lep_severity=self._get_lep_severity() if lep_injected else None,
            )

            action = self._select_action(task, step, current_agent, history)

            if step >= MIN_EVENTS_PER_RUN and action in ["read_document", "search_notes"]:
                action = "final"

            if action == "final":
                final_response = f"Collaboration complete. {current_agent.name} synthesizing findings from {len(history)} observations."

                collector.log_event(
                    event_type="llm_output",
                    source=current_agent.agent_id,
                    target="internal",
                    input_summary=task,
                    output_summary=final_response,
                    expected_behavior=f"{current_agent.role} produces final synthesis.",
                    observed_behavior=f"{current_agent.agent_id} concluded with synthesis.",
                    agent_id=current_agent.agent_id,
                    agent_name=current_agent.name,
                )

                collector.log_event(
                    event_type="final_response",
                    source=current_agent.agent_id,
                    target="user",
                    input_summary=task,
                    output_summary=final_response,
                    expected_behavior="Collaborative answer returned to user.",
                    observed_behavior=f"Collaboration complete. {handoff_count} handoffs.",
                    agent_id=current_agent.agent_id,
                    agent_name=current_agent.name,
                    
                    lep_injected=lep_injected_this_run,
                    downstream_failure=lep_injected_this_run and random.random() < 0.5,
                    failure_type="wrong_answer" if lep_injected_this_run else None,
                )
                break

            tool = self.tools.get(action)
            if tool is None:
                action = "final"
                continue

            self._log_tool_event(
                collector, "tool_call",
                current_agent, tool,
                input_summary=f"{task}",
                output_summary="",
                expected_behavior=f"{current_agent.name} invokes {tool.name}.",
                observed_behavior=f"{current_agent.agent_id} called {tool.tool_id} ({tool.name}).",
                lep_injected=lep_injected,
                lep_type="2.3 Tool Corruption" if lep_injected else None,
            )

            if lep_injected and hasattr(tool.func, '__code__'):
                tool_output = tool.func(tool.name, lep_injected=True)
            else:
                tool_output = tool.func(tool.name)

            self._log_tool_event(
                collector, "tool_result",
                current_agent, tool,
                input_summary=f"{tool.name} input",
                output_summary=tool_output["output"],
                expected_behavior=f"Tool {tool.name} returns result.",
                observed_behavior=f"Tool {tool.tool_id} returned to {current_agent.agent_id}.",
                lep_injected=lep_injected,
                lep_type="3.1 Output Corruption" if lep_injected else None,
                lep_category="FC2" if lep_injected else None,
            )

            history.append({
                "agent_id": current_agent.agent_id,
                "agent_name": current_agent.name,
                "tool_id": tool.tool_id,
                "tool": tool.name,
                "input": action,
                "output": tool_output["output"],
            })

            time.sleep(0.01)

        else:
            final_response = f"Max steps reached. {handoff_count} handoffs, {len(history)} observations."
            collector.log_event(
                event_type="final_response",
                source=primary_agent.agent_id,
                target="user",
                input_summary=task,
                output_summary=final_response,
                expected_behavior="Partial results at step limit.",
                observed_behavior="Max steps reached.",
                
                lep_injected=lep_injected_this_run,
                downstream_failure=True,
                failure_type="non_termination",
            )

        return {
            "trace_id": collector.trace_id,
            "run_id": run_id,
            "trace_type": trace_type,
            "final_response": final_response,
            "num_events": len(collector.events),
            "path": str(collector.path),
            "lep_injected": lep_injected_this_run,
            "handoff_count": handoff_count,
            "primary_agent": primary_agent.agent_id,
            "secondary_agent": secondary_agent.agent_id,
            "events": collector.events
        }

    def _generate_reasoning_content(self, task: str, step: int, history: List, agent: AgentConfig) -> str:
        """Generate reasoning content based on agent role and step."""
        if agent.name == "researcher":
            if step == 1:
                return f"{agent.name} gathering initial information for the task."
            elif step < 5:
                return f"{agent.name} searching for relevant documents and data."
            else:
                return f"{agent.name} cross-referencing findings."
        else:
            if "calculate" in task.lower() or "roi" in task.lower() or "profit" in task.lower():
                return f"{agent.name} performing financial calculations."
            else:
                return f"{agent.name} analyzing data patterns."

    def _select_action(self, task: str, step: int, agent: AgentConfig, history: List) -> str:
        """Select appropriate action based on agent capabilities and task."""
        if agent.name == "analyst":
            if "calculate" in task.lower() or "roi" in task.lower() or "profit" in task.lower():
                if step < 10:
                    return "calculator"

        if agent.name == "researcher":
            if step == 1:
                if "read" in task.lower():
                    return "read_document"
                elif "search" in task.lower():
                    return "search_notes"

        if step < 10:
            return random.choice(["read_document", "search_notes", "analyze_data"])
        elif step < 25:
            return random.choice(["analyze_data", "calculator", "read_document"])
        else:
            return random.choice(["analyze_data", "coordinate_agents"])

    def generate_trace_pair(self, task: str, run_id: str) -> Dict[str, Dict[str, Any]]:
        """Generate both benign and malignant traces for a task."""
        print(f"Generating trace pair for run {run_id}...")

        benign_result = self._generate_benign_trace(task, run_id)
        malignant_result = self._generate_malignant_trace(task, run_id)

        return {
            "benign": benign_result,
            "malignant": malignant_result
        }

    def generate_dataset(self, num_runs: int = 5) -> Dict[str, List[Dict[str, Any]]]:
        """Generate a dataset of trace pairs."""

        dataset = {
            "traces": [],
            "metadata": {
                "num_runs": num_runs,
                "min_events_per_run": MIN_EVENTS_PER_RUN,
                "max_events_per_run": MAX_EVENTS_PER_RUN,
                "lep_injection_rate": LEP_INJECTION_RATE,
                "agents": [{"agent_id": a.agent_id, "name": a.name, "role": a.role} for a in AGENTS],
                "generated_at": datetime.now(timezone.utc).isoformat()
            }
        }

        for i in range(num_runs):
            run_id = f"run_{i+1:03d}_{uuid.uuid4().hex[:8]}"
            task = random.choice(TASK_POOL)
            trace_pair = self.generate_trace_pair(task, run_id)

            dataset["traces"].append({
                "run_id": run_id,
                "task": task,
                "benign": trace_pair["benign"],
                "malignant": trace_pair["malignant"]
            })

            print(f"Completed run {i+1}/{num_runs}: {run_id} | "
                  f"Benign: {trace_pair['benign']['num_events']} events | "
                  f"Malignant: {trace_pair['malignant']['num_events']} events")

        metadata_path = self.trace_dir / "dataset_metadata.json"
        with metadata_path.open("w", encoding="utf-8") as f:
            json.dump(dataset["metadata"], f, indent=2)

        return dataset


print("Initializing multi-agent system with collaboration...")
multi_agent_system = MultiAgentSystem(llm_backend, ENHANCED_TOOLS)


Initializing multi-agent system with collaboration...


In [10]:
# --- Main Execution ---
if __name__ == "__main__":
    print("\n=== Enhanced Multi-Agent Benchmarking System ===\n")
    
    # Generate dataset
    num_runs = 3  # Start with 3 for testing
    dataset = multi_agent_system.generate_dataset(num_runs=num_runs)
    
    print(f"\n=== Dataset Summary ===")
    print(f"Total runs: {len(dataset['traces'])}")
    print(f"Total traces: {len(dataset['traces']) * 2}")
    print(f"Min events per trace: {MIN_EVENTS_PER_RUN}")
    print(f"Max events per trace: {MAX_EVENTS_PER_RUN}")
    
    # Analyze dataset
    total_events = 0
    lep_count = 0
    
    for run_data in dataset["traces"]:
        benign = run_data["benign"]
        malignant = run_data["malignant"]
        
        total_events += benign["num_events"] + malignant["num_events"]
        lep_count += int(malignant["lep_injected"])
        
        print(f"\nRun {run_data['run_id']}:")
        print(f"  Task: {run_data['task'][:60]}...")
        print(f"  Benign: {benign['num_events']} events")
        print(f"  Malignant: {malignant['num_events']} events (LEP: {malignant['lep_injected']})")
    
    print(f"\n=== Statistics ===")
    print(f"Total events generated: {total_events}")
    print(f"Average events per trace: {total_events / (len(dataset['traces']) * 2):.1f}")
    print(f"Traces with LEPs: {lep_count}/{len(dataset['traces'])}")
    
    # Save dataset summary
    summary = {
        "total_runs": len(dataset["traces"]),
        "total_traces": len(dataset["traces"]) * 2,
        "total_events": total_events,
        "lep_traces": lep_count,
        "avg_events_per_trace": total_events / (len(dataset["traces"]) * 2),
        "dataset_path": str(TRACE_DIR),
        "generated_at": datetime.now(timezone.utc).isoformat()
    }
    
    summary_path = TRACE_DIR / "dataset_summary.json"
    with summary_path.open("w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    
    print(f"\nDataset summary saved to: {summary_path}")
    print(f"All traces saved to: {TRACE_DIR.resolve()}")


=== Enhanced Multi-Agent Benchmarking System ===

Generating trace pair for run run_001_26760fce...
Completed run 1/3: run_001_26760fce | Benign: 156 events | Malignant: 156 events
Generating trace pair for run run_002_e62217eb...
Completed run 2/3: run_002_e62217eb | Benign: 156 events | Malignant: 156 events
Generating trace pair for run run_003_7d83d9af...
Completed run 3/3: run_003_7d83d9af | Benign: 156 events | Malignant: 156 events

=== Dataset Summary ===
Total runs: 3
Total traces: 6
Min events per trace: 90
Max events per trace: 120

Run run_001_26760fce:
  Task: Read the financial report and calculate profit margins for Q...
  Benign: 156 events
  Malignant: 156 events (LEP: True)

Run run_002_e62217eb:
  Task: Calculate the average revenue growth rate over the last 5 qu...
  Benign: 156 events
  Malignant: 156 events (LEP: True)

Run run_003_7d83d9af:
  Task: Read competitive analysis and summarize key insights....
  Benign: 156 events
  Malignant: 156 events (LEP: True)



In [11]:
# --- Enhanced Visualization ---
print("\n=== Visualization ===\n")

# Load and analyze the generated traces
all_traces = []
trace_types = []

for run_data in dataset["traces"]:
    # Load benign trace
    benign_path = Path(run_data["benign"]["path"])
    if benign_path.exists():
        benign_events = TraceCollector.load_jsonl(benign_path)
        all_traces.append({"run_id": run_data["run_id"], "type": "benign", "events": benign_events})

    # Load malignant trace
    malignant_path = Path(run_data["malignant"]["path"])
    if malignant_path.exists():
        malignant_events = TraceCollector.load_jsonl(malignant_path)
        all_traces.append({"run_id": run_data["run_id"], "type": "malignant", "events": malignant_events})

# Create comprehensive analysis
print(f"Analyzing {len(all_traces)} traces...")

# Convert to DataFrame for analysis with new fields
df_data = []
for trace in all_traces:
    for event in trace["events"]:
        df_data.append({
            "run_id": trace["run_id"],
            "trace_type": trace["type"],
            "event_id": event["event_id"],
            "timestamp": event["timestamp"],
            "event_type": event["event_type"],
            "source": event["source"],
            "target": event["target"],
            "agent_id": event.get("agent_id", ""),
            "agent_name": event.get("agent_name", ""),
            "agent_role": event.get("agent_role", ""),
            "tool_id": event.get("tool_id", ""),
            "tool_name": event.get("tool_name", ""),
            "lep_injected": event.get("lep_injected", False),
            "lep_type": event.get("lep_type"),
            "lep_severity": event.get("lep_severity"),
            "downstream_failure": event.get("downstream_failure", False),
            "failure_type": event.get("failure_type"),
        })

df = pd.DataFrame(df_data)

if len(df) > 0:
    # Display trace comparison
    print("\n=== Trace Comparison ===")
    trace_stats = df.groupby(["run_id", "trace_type"]).agg({
        "event_id": "count",
        "lep_injected": "sum",
        "downstream_failure": "sum"
    }).rename(columns={"event_id": "num_events"})

    print(trace_stats)

    # Agent activity analysis
    print("\n=== Agent Activity ===")
    if "agent_id" in df.columns:
        agent_stats = df[df["agent_id"] != ""].groupby(["run_id", "trace_type", "agent_id", "agent_name"]).size().reset_index(name="event_count")
        print(agent_stats)

    # Tool usage analysis
    print("\n=== Tool Usage by Agent ===")
    if "tool_id" in df.columns:
        tool_stats = df[df["tool_id"] != ""].groupby(["agent_name", "tool_id", "tool_name"]).size().reset_index(name="call_count")
        print(tool_stats)

    # Event type distribution
    print("\n=== Event Type Distribution ===")
    event_counts = df.groupby(["trace_type", "event_type"]).size().unstack(fill_value=0)
    print(event_counts)

    # LEP analysis
    print("\n=== LEP Analysis ===")
    lep_traces = df[df["lep_injected"]].groupby("run_id").size()
    print(f"Traces with LEPs: {len(lep_traces)}")
    if len(lep_traces) > 0:
        print("LEP distribution by type:")
        lep_types = df[df["lep_injected"]]["lep_type"].value_counts()
        print(lep_types)

        print("\nLEP severity distribution:")
        lep_severity = df[df["lep_injected"]]["lep_severity"].value_counts()
        print(lep_severity)

        print("\nLEP affected agents:")
        lep_agents = df[df["lep_injected"]]["agent_name"].value_counts()
        print(lep_agents)

    # Agent handoff analysis
    print("\n=== Agent Handoff Analysis ===")
    handoff_events = df[df["event_type"] == "agent_handoff"]
    if len(handoff_events) > 0:
        print(f"Total handoffs: {len(handoff_events)}")
        handoff_by_run = handoff_events.groupby(["run_id", "trace_type"]).size()
        print(handoff_by_run)

    # Failure analysis
    print("\n=== Failure Analysis ===")
    failures = df[df["downstream_failure"]]
    if len(failures) > 0:
        print("Failures by type:")
        failure_types = failures["failure_type"].value_counts()
        print(failure_types)

        print("\nFailure analysis by trace type:")
        failure_by_type = failures.groupby(["trace_type", "failure_type"]).size()
        print(failure_by_type)

    # Save processed dataset
    processed_path = TRACE_DIR / "processed_dataset.csv"
    df.to_csv(processed_path, index=False)
    print(f"\nProcessed dataset saved to: {processed_path}")

    # Visualization examples
    print("\n=== Visualization Examples ===")

    # 1. Event timeline for first run with agent IDs
    print(f"\n1. Event Timeline for Run {df['run_id'].iloc[0]}:")
    run_events = df[(df["run_id"] == df["run_id"].iloc[0]) & (df["trace_type"] == "benign")].sort_values("event_id")
    for _, event in run_events.head(15).iterrows():
        agent_info = f"[{event['agent_id']}/{event['agent_name']}]" if event['agent_id'] else ""
        tool_info = f" -> [{event['tool_id']}/{event['tool_name']}]" if event['tool_id'] else ""
        print(f"  #{event['event_id']:3d} {event['event_type']:15} {event['source']:12} -> {event['target']:12} {agent_info}{tool_info} [LEP: {event['lep_injected']}]")

    # 2. Agent collaboration example
    print(f"\n2. Agent Collaboration Example:")
    handoffs = df[(df["run_id"] == df["run_id"].iloc[0]) & (df["event_type"] == "agent_handoff")]
    if len(handoffs) > 0:
        for _, h in handoffs.iterrows():
            print(f"  Handoff: {h['agent_id_from']} ({h['agent_name_from']}) -> {h['agent_id_to']} ({h['agent_name_to']})")
    else:
        print("  No handoffs in this trace")

    # 3. LEP propagation example
    print(f"\n3. LEP Propagation Example:")
    lep_events = df[(df["run_id"] == df["run_id"].iloc[0]) & df["lep_injected"]].sort_values("event_id")
    if len(lep_events) > 0:
        print("  Events with LEPs:")
        for _, event in lep_events.head(5).iterrows():
            print(f"    #{event['event_id']}: {event['lep_type']} (severity: {event['lep_severity']}) - Agent: {event['agent_name']}")

    # 4. Comparison of benign vs malignant
    print(f"\n4. Benign vs Malignant Comparison:")
    comparison = df.groupby("trace_type").agg({
        "event_id": "count",
        "lep_injected": "sum",
        "downstream_failure": "sum"
    }).rename(columns={"event_id": "total_events"})
    print(comparison)
else:
    print("No traces found for visualization.")



=== Visualization ===

Analyzing 6 traces...

=== Trace Comparison ===
                             num_events  lep_injected  downstream_failure
run_id           trace_type                                              
run_001_26760fce benign             156             0                   1
                 malignant          156            43                   1
run_002_e62217eb benign             156             0                   1
                 malignant          156            34                   1
run_003_7d83d9af benign             156             0                   1
                 malignant          156            31                   1

=== Agent Activity ===
              run_id trace_type   agent_id  agent_name  event_count
0   run_001_26760fce     benign  agent_001  researcher          141
1   run_001_26760fce     benign  agent_002     analyst            9
2   run_001_26760fce  malignant  agent_001  researcher          141
3   run_001_26760fce  malignant  agent_0

KeyError: 'agent_id_from'

In [ ]:
# Improved Multi-Agent Benchmarking System
# Features:
# 1. Robust LEP injection mechanism
# 2. No fallbacks - requires real LLM
# 3. Multi-agent support (2 agents)
# 4. Comprehensive traces (~100 events)
# 5. Benign and malignant trace pairs with run IDs

# This notebook implements a comprehensive benchmarking system for multi-agent failures
# with Localized Execution Perturbations (LEPs).

In [8]:
pip install pandas torch transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 113.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.1/765.1 kB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 801.3/801.3 kB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 118.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.3/78.3 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.6/122.6 kB 37.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 117.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.7/310.7 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━